# Transcoder-based Explainability for InterveneEncoder

This notebook demonstrates a lightweight **explainability layer** for the bidirectional EMR encoder defined in `intervene_enc/transformer.py`.

Each encoder block contains a **SwiGLU MLP** of the form
$$\text{MLP}(x) = W_2 \,\bigl(x_{\text{val}} \odot \text{SiLU}(x_{\text{gate}})\bigr), \qquad [x_{\text{val}};\, x_{\text{gate}}] = W_1 x.$$
The non-linear gating makes the internals hard to read directly. We side-step this by training a **JumpReLU transcoder** per layer – a sparse dictionary learner that approximates the *input → output* map of the MLP through a high-dimensional but sparsely-active feature space:
$$\hat{y} \;=\; W_{\text{dec}}\,f(x) + b_{\text{dec}}, \qquad f(x) = \text{JumpReLU}(W_{\text{enc}} x + b_{\text{enc}};\, \theta).$$
Because we regress directly on the I/O pair, we never have to decompose the SwiGLU non-linearity.

What the notebook shows:
1. Capture MLP I/O activations from a frozen Phase-3 checkpoint.
2. Train one JumpReLU transcoder per layer (a few minutes on CPU for the M-128 model).
3. **Reconstruction fidelity** – MSE / explained-variance per layer.
4. **Sparsity** – L0 distribution across tokens; only a handful of features are active per token.
5. **Feature-firing heatmap** – which features fire across the patient timeline.
6. **Logit attribution** – for one (patient, outcome), which (layer, token, feature) triples drove the prediction.

*Goal: demonstrate the capability rigorously enough to be convincing, while staying small enough to run end-to-end in the notebook.*

In [ ]:
import os, sys, math, copy
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# The notebook lives in `xai/`; project root is one level up.
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'font.size': 10,
})
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device = {DEVICE}')

# --------------------------------------------------------------------- #
# Reuse-from-disk knobs.  Both default ON so re-runs of the notebook
# skip the multi-minute capture / training steps when the artefacts
# from a previous run are already on disk under `checkpoints/`.
#   * activations cache:  checkpoints/activations_cache/layer_{i}.pt
#   * transcoders bundle: checkpoints/transcoders.pt
# Flip either flag to False to force a fresh build of that artefact.
# --------------------------------------------------------------------- #
REUSE_ACTIVATIONS_CACHE = True
REUSE_TRANSCODERS       = True

ACTIVATIONS_CACHE_DIR = PROJECT_ROOT / 'checkpoints' / 'activations_cache'
TRANSCODERS_PATH      = PROJECT_ROOT / 'checkpoints' / 'transcoders.pt'


def clear_mlp_hooks(model):
    """
    Purpose: Remove every forward / forward-pre hook from each block's MLP.
    Method:  Iterates `model.blocks[i].mlp` and clears the OrderedDicts that
             back `register_forward_hook` / `register_forward_pre_hook`.
    Why:     Notebook re-runs without a kernel restart can leave stale
             TranscoderHookManager hooks attached on `block.mlp` if a prior
             cell errored out before reaching its `.detach()`.  Those stale
             hooks then silently corrupt the baseline (§6 bit-exactness)
             and attribution (§7) cells, producing spurious large Δ logit
             gaps that look like a transcoder failure but are actually a
             dangling-hook artefact.  Call this at the top of §6 and §7.
    """
    n_removed = 0
    for blk in model.blocks:
        mlp = blk.mlp
        n_removed += len(mlp._forward_hooks) + len(mlp._forward_pre_hooks)
        mlp._forward_hooks.clear()
        mlp._forward_pre_hooks.clear()
    print(f'cleared {n_removed} stale MLP hook(s) across {len(model.blocks)} blocks.')
    return n_removed

## 1. Load the frozen encoder and a small dataloader

We rebuild the embedder + encoder exactly as the Phase-3 pipeline does and load the best Phase-3 checkpoint (with task heads). Everything from here on is read-only on the encoder.

In [ ]:
from intervene_enc import (
    DataProcessor, EMRDataset, EMRTokenizer, get_dataloader,
    EMREmbedding, InterveneEncoder,
)
from intervene_enc.config.model_config import (
    MODEL_CONFIG, PHASE1_CHECKPOINT, PHASE3_CHECKPOINT,
)

# --- build embedder + encoder from config ---
tokenizer = EMRTokenizer.load()                      # uses default checkpoint dir
# Load the Phase-1 embedder from its own checkpoint -- this recovers ctx_dim
# / time2vec_dim / embed_dim from the saved config, so we don't have to
# guess the context-vector width.
embedder, *_ = EMREmbedding.load(PHASE1_CHECKPOINT, tokenizer=tokenizer,
                                 map_location=DEVICE)

# Load Phase-3 ckpt; task heads come along for attribution.
model, *_ = InterveneEncoder.load(
    PHASE3_CHECKPOINT, embedder=embedder, map_location=DEVICE,
    attach_task_heads=True,
)
model = model.to(DEVICE).eval()
for p in model.parameters():
    p.requires_grad_(False)
print(f'model loaded: {model.get_num_params()/1e6:.2f} M params, {len(model.blocks)} blocks, d={MODEL_CONFIG["embed_dim"]}')

In [ ]:
from intervene_enc import collate_emr
from torch.utils.data import DataLoader
import random, gc

# --- 1. Load source CSVs (aligned with the tak-repo + tokenizer the ckpt was trained on) ---
train_temporal = pd.read_csv(PROJECT_ROOT / 'data' / 'source' / 'temporal_data.csv')
train_ctx      = pd.read_csv(PROJECT_ROOT / 'data' / 'source' / 'context_data.csv')

# --- 2. Process + build the full dataset once so we can stratify on outcome membership ---
proc = DataProcessor(train_temporal, train_ctx,
                     tak_repo_path=str(PROJECT_ROOT / 'intervene_enc' / 'config' / 'tak-repo-portable.json'))
proc.run()
full_ds = EMRDataset(proc.df, proc.context_df, tokenizer)

# --- 3. Stratified patient subset --------------------------------------------- #
# Scaled up from 250 -> 1500 patients so we have ~1k tokens per dictionary feature
# at EXPANSION=16.  RAM is kept in check because activations are spilled to disk
# in bf16 (see next cell), so the dense full-layer tensors never live in memory.
N_PATIENTS        = 1500
PER_OUTCOME_QUOTA = 150           # patients pulled from each outcome's positives.

rng = random.Random(0)
outcome_names = list(model.outcome_names)

print('scanning patient timelines for outcome membership...')
pid_to_concepts = {pid: set(g['Concept'].values) for pid, g in full_ds.patient_groups.items()}
all_pids = list(full_ds.patient_ids)

selected = set()
for oc in outcome_names:
    pos_pids = [pid for pid, cs in pid_to_concepts.items() if oc in cs]
    take = min(PER_OUTCOME_QUOTA, len(pos_pids))
    if take == 0:
        print(f'  [warn] outcome "{oc}" has 0 positive patients in source data.')
        continue
    chosen = rng.sample(pos_pids, take)
    selected.update(chosen)
    print(f'  {oc:35s}  positives in source: {len(pos_pids):6d}  -> took {take}')

quota_n  = len(selected)
remaining = [p for p in all_pids if p not in selected]
fill_n   = max(0, N_PATIENTS - quota_n)
selected.update(rng.sample(remaining, min(fill_n, len(remaining))))
selected_list = list(selected)
print(f'final subset: {len(selected_list)} patients '
      f'({quota_n} from outcome quotas + {fill_n} random fill)')

# --- 4. Build the subset dataset, then free the heavy intermediates ----------- #
subset_tokens_df  = full_ds.tokens_df[full_ds.tokens_df['PatientId'].isin(selected_list)].copy()
subset_context_df = full_ds.context_df.loc[full_ds.context_df.index.isin(selected_list)].copy()
ds = EMRDataset(subset_tokens_df, subset_context_df, tokenizer)

# Deterministic loader for §6 (bit-exactness) and §7 (attribution): the
# project's `get_dataloader` defaults to shuffle=True, so two iterations
# return patients in different orders and a naive risk_real vs risk_decoupled
# comparison is comparing outputs on *different patients* (we saw Δ logit
# ≈ 16.5 between two no-hook passes on this dataloader -- same magnitude as
# the apparent transcoder gap, which is the actual root cause).
#
# Going through torch's DataLoader directly here (shuffle=False, num_workers=0)
# guarantees a stable order across iterations.  This is the loader §2/§6/§7
# all use; activations capture is invariant to order so the same loader is
# reused for the (now-skipped, on-disk) capture step.
dl = DataLoader(
    ds, batch_size=8, shuffle=False, collate_fn=collate_emr,
    num_workers=0, pin_memory=False,
)
print(f'subset dataset: {len(ds)} patients, {len(dl)} batches at batch_size=8  '
      '(deterministic: shuffle=False, num_workers=0)')

# Drop the multi-GB pre-subset structures -- we won't need them again.
del full_ds, pid_to_concepts, proc, train_temporal, train_ctx
gc.collect()
print('freed full dataset + raw dataframes.')

## 2. Capture MLP I/O activations

We register forward hooks on every `block.mlp` and run a handful of batches through the frozen encoder. Padded positions are filtered out so the transcoder never sees `[PAD]` rows.

In [ ]:
from xai.transcoder.hooks import collect_activations_cached, load_layer_cache

# Spill activations to disk one layer at a time, in bf16.  At ~1500 patients and
# 4 layers the dense fp32 form would be ~8-12 GB; with this routine peak RAM is
# bounded by a single layer's bf16 tensor (~1-1.5 GB) and the rest lives on disk
# as memory-mappable .pt files.
#
# When REUSE_ACTIVATIONS_CACHE is True, `overwrite=False` makes the helper
# short-circuit if all layer_*.pt files already exist -- no encoder forward
# is run at all, the function just returns the existing paths.
CACHE_DIR = ACTIVATIONS_CACHE_DIR
cache_files_present = (
    CACHE_DIR.exists()
    and all((CACHE_DIR / f'layer_{i}.pt').exists() for i in range(len(model.blocks)))
)
if REUSE_ACTIVATIONS_CACHE and cache_files_present:
    print(f'[reuse] using activations cache under {CACHE_DIR}')
    activations = {i: CACHE_DIR / f'layer_{i}.pt' for i in range(len(model.blocks))}
else:
    if REUSE_ACTIVATIONS_CACHE and not cache_files_present:
        print('[reuse] cache not complete on disk -- running a fresh capture.')
    activations = collect_activations_cached(
        model, dataloader=dl, device=DEVICE,
        cache_dir=CACHE_DIR,
        store_dtype=torch.bfloat16,
        overwrite=True,
    )

for i, p in activations.items():
    X_peek, _ = load_layer_cache(p, mmap=True)
    print(f'layer {i}: cache={p.name}  shape={tuple(X_peek.shape)}  dtype={X_peek.dtype}')
    del X_peek
gc.collect()

## 3. Train one transcoder per layer

* Dictionary expansion factor: **8×** d_model. With d=128 this gives 1024 features per layer – a reasonable starting point.
* Loss = MSE(reconstruction) + λ·L0(features), with JumpReLU's straight-through estimator on the threshold gate.
* Train for a few epochs on CPU/GPU; converges fast at this scale.

In [ ]:
from xai.transcoder.train import train_transcoders, save_transcoders, load_transcoders

D = MODEL_CONFIG['embed_dim']
EXPANSION = 16                  # 16x dictionary (2048 features at d=128) -- was 4x.
N_FEATURES = EXPANSION * D

# Reuse the trained-and-saved bundle when present and the flag is on -- skips
# the (slow) training loop entirely on warm re-runs.  `hist` is None in that
# case; the next cell handles that and just notes "loaded from disk".
if REUSE_TRANSCODERS and TRANSCODERS_PATH.exists():
    print(f'[reuse] loading transcoders from {TRANSCODERS_PATH}')
    transcoders = load_transcoders(TRANSCODERS_PATH, map_location='cpu')
    hist = None
    print(f'  loaded {len(transcoders)} transcoders '
          f'(d_model={D}, n_features={N_FEATURES})')
else:
    # Notes on the training schedule:
    #   * lambda_l0 ramps linearly 0 -> 5e-5 over the first 20% of steps so MSE can
    #     settle before sparsity pressure starts killing features (the old run at
    #     2e-4 from step 0 with init_theta=0.05 looked like cold-start feature death).
    #   * LR: linear warmup 5% -> cosine decay to 10% of peak.  Helps the late epochs
    #     stop overshooting on this tiny model.
    #   * init_theta=0.01 (was 0.05) keeps the JumpReLU gate near zero at init so
    #     features are alive before lambda_l0 ramps up.
    transcoders, hist = train_transcoders(
        activations,                # dict of cache paths -> trained lazily via mmap
        d_model=D, n_features=N_FEATURES,
        n_epochs=60, batch_size=4096,
        lr=3e-4, lr_min_factor=0.1, lr_warmup_frac=0.05,
        lambda_l0=5e-5, lambda_warmup_frac=0.2,
        init_theta=0.01,
        device=str(DEVICE),
    )
    save_transcoders(transcoders, TRANSCODERS_PATH)
    print(f'trained + saved transcoders to {TRANSCODERS_PATH}')

### Training curves
MSE should fall steadily; L0 (mean features active per token) should decay toward a small fraction of `n_features`.

In [ ]:
# When `transcoders` came from disk (REUSE_TRANSCODERS=True path), there is
# no training history to plot -- skip gracefully so a warm re-run is silent.
if not hist:
    print('[reuse] transcoders loaded from disk -- no training history to plot.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
    for i, h in hist.items():
        axes[0].plot(h['mse'], label=f'L{i}')
        axes[1].plot(h['l0'],  label=f'L{i}')
    axes[0].set_title('Reconstruction MSE'); axes[0].set_xlabel('epoch'); axes[0].set_yscale('log')
    axes[1].set_title('Active features / token (L0)'); axes[1].set_xlabel('epoch')
    for ax in axes:
        ax.legend(frameon=False, fontsize=8)
    plt.tight_layout(); plt.show()

## 4. Reconstruction fidelity

**Explained variance** (R²) of the transcoder reconstruction vs. the real MLP output, per layer. If the bars sit near 1, the sparse approximation is faithful and any conclusion drawn through the transcoder is a conclusion about the real model.

In [ ]:
@torch.no_grad()
def explained_variance(tc, X, Y, batch=8192):
    """R^2 = 1 - var(Y - Y_hat) / var(Y), micro-averaged over coordinates."""
    ss_res = 0.0; ss_tot = 0.0
    y_mean = Y.float().mean(0, keepdim=True)
    for s in range(0, X.shape[0], batch):
        xb = X[s:s+batch].float().to(DEVICE)
        yb = Y[s:s+batch].float().to(DEVICE)
        _, yh = tc(xb)
        ss_res += ((yb - yh) ** 2).sum().item()
        ss_tot += ((yb - y_mean.to(DEVICE)) ** 2).sum().item()
    return 1.0 - ss_res / ss_tot

# Move each transcoder to device just-in-time, score against its mmap'd cache,
# then move back -- keeps GPU memory flat across layers.
r2 = {}
for i, tc in transcoders.items():
    X, Y = load_layer_cache(activations[i], mmap=True)
    tc_d = tc.to(DEVICE)
    r2[i] = explained_variance(tc_d, X, Y)
    tc.cpu()
    del X, Y; gc.collect()

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.bar(list(r2.keys()), list(r2.values()), color='#3a86ff')
ax.set_ylim(0, 1); ax.set_ylabel('R²'); ax.set_xlabel('encoder layer')
ax.set_title('Transcoder reconstruction fidelity vs. real MLP output')
for i, v in r2.items():
    ax.text(i, v + 0.01, f'{v:.2f}', ha='center', fontsize=9)
plt.tight_layout(); plt.show()

## 5. Sparsity: how many features fire per token?

A useful sparse code has *many* features that are dormant on any given token and only a handful that fire. The per-token L0 distribution should sit far below the dictionary size.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.4))
for i, tc in transcoders.items():
    X, _ = load_layer_cache(activations[i], mmap=True)
    tc_d = tc.to(DEVICE)
    with torch.no_grad():
        f = tc_d.encode(X[:8000].float().to(DEVICE))
    tc.cpu()
    l0_per_tok = (f > 0).sum(dim=-1).cpu().numpy()
    ax.hist(l0_per_tok, bins=40, alpha=0.5, label=f'L{i} (mean={l0_per_tok.mean():.1f})')
    del X; gc.collect()
ax.set_xlabel('# active features per token'); ax.set_ylabel('token count')
ax.set_title(f'Per-token L0 across layers   (dictionary size = {N_FEATURES})')
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
# Drop the path dict -- §6+ re-derive features via hooks one layer at a time,
# so we no longer need the cache references.  The cache files on disk remain
# (under checkpoints/activations_cache/) for reuse on subsequent runs.
import gc
del activations
gc.collect()
print('activations dict freed (on-disk caches preserved).')

## 6. Attribution methodology: gradient-decoupled transcoder probe

**The problem with the obvious approach.** A first instinct is to *swap* each block's MLP for its transcoder reconstruction and explain the swapped model. That works only if the swap leaves the risk logits intact. It doesn't: at per-layer R² ≈ 0.97 (§4) the reconstruction error is small in MLP-output units but the head amplifies it sharply, so even a **single-layer** swap collapses the risk logits' rank correlation against the real model to ρ ≈ 0.02. Attributions computed against a swapped model would describe a surrogate that doesn't behave like the deployed one.

**What we do instead.** A *gradient-decoupled* hook. At every block we compute the transcoder features $f_L = \mathrm{tc}_L.\text{encode}(x_L)$ and its reconstruction $\hat y_L = \mathrm{tc}_L.\text{decode}(f_L)$, but inject

$$y^{\text{used}}_L \;=\; y^{\text{true}}_L + (\hat y_L - \mathrm{sg}(\hat y_L))$$

into the residual stream, where $\mathrm{sg}(\cdot)$ is stop-gradient. The bracket is **numerically zero** so the forward pass — and therefore the risk logits — are bit-exact equal to the deployed model. But autograd still flows from the risk logit through $\hat y_L$ to $f_L$, giving a well-defined $\partial \mathrm{risk}_k / \partial f_L$.

**Attribution.** First-order decomposition through the unmodified model:

$$\text{contrib}_t^{(k)} \;=\; \sum_{L=1}^{4} f_L(t) \;\cdot\; \frac{\partial \mathrm{risk}_k}{\partial f_L(t)}\,.$$

Nothing in this expression depends on swap fidelity. The transcoder's role is exactly what we trained it for — a sparse, named-concept basis for each MLP output (§4 R² ≈ 0.97 justifies treating it as such). The gradient × activation product gives the linearized contribution of each (layer, token, feature) triple to the *deployed model's* prediction.

The cell below is a sanity check: it runs `model.predict` with the gradient-decoupled hooks attached and confirms the risk logits are bit-exact equal to the un-hooked model.

In [ ]:
from xai import TranscoderHookManager
from tqdm.auto import tqdm

# Belt-and-braces: nuke any hooks left behind by a prior cell run.  The
# `y_used = y + (y_hat - y_hat.detach())` identity is algebraically zero in
# floating point, so a non-bit-exact result here was previously caused by
# *stale* hooks from an earlier execution (the cell errored partway and
# never reached `mgr.detach()`), not by a transcoder fidelity problem.
clear_mlp_hooks(model)

@torch.no_grad()
def _collect_risk(model, dl, mgr_or_none, desc):
    """Run dl through model.predict, optionally with a manager attached."""
    if mgr_or_none is not None:
        mgr_or_none.attach()
    outs = []
    try:
        for b in tqdm(dl, desc=desc, leave=False):
            b = {k: v.to(DEVICE) for k, v in b.items()}
            risk, *_ = model.predict(**{k: b[k] for k in
                ['parent_raw_ids','concept_ids','value_ids',
                 'position_ids','abs_ts','context_vec']})
            outs.append(risk.cpu())
    finally:
        if mgr_or_none is not None:
            mgr_or_none.detach()
    return torch.cat(outs, dim=0)


# Self-consistency check: running the baseline twice MUST be bit-exact.  If
# it isn't, the dataloader is non-deterministic and the §6 comparison would
# be meaningless regardless of what the hook does.  Catch that first.
print('--- baseline determinism check (two no-hook passes) ---')
risk_real    = _collect_risk(model, dl, None, 'predict[real-1]')
risk_real_v2 = _collect_risk(model, dl, None, 'predict[real-2]')
det_err = (risk_real - risk_real_v2).abs().max().item()
print(f'  max |Δ logit| between two no-hook runs = {det_err:.3e}')
assert det_err < 1e-5, (
    f'Dataloader is non-deterministic (Δ={det_err:.3e}). '
    'Re-instantiate `dl` with shuffle=False / a fixed seed before comparing.'
)

print('--- with gradient-decoupled hooks attached ---')
mgr = TranscoderHookManager(model, transcoders,
                            gradient_decoupled=True, enabled=True)
risk_decoupled = _collect_risk(model, dl, mgr, 'predict[decoupled]')

# Bit-exactness check.  (y_hat - y_hat.detach()) is numerically zero so the
# decoupled forward must match the real forward to floating-point precision.
abs_err = (risk_real - risk_decoupled).abs()
max_err = abs_err.max().item()
rmse    = abs_err.pow(2).mean().sqrt().item()
print(f'  max |Δ logit|  = {max_err:.3e}')
print(f'  RMSE Δ logit   = {rmse:.3e}')
print(f'  -> forward is {"bit-exact" if max_err < 1e-6 else "NOT bit-exact"} '
      f'equal to the deployed model.')

fig, ax = plt.subplots(figsize=(4.4, 4.4))
lim = max(risk_real.abs().max().item(),
          risk_decoupled.abs().max().item()) * 1.05
ax.scatter(risk_real.flatten(), risk_decoupled.flatten(),
           s=10, alpha=0.4, color='#3a86ff')
ax.plot([-lim, lim], [-lim, lim], 'k--', lw=0.8)
ax.set_xlabel('risk logit (real, no hooks)')
ax.set_ylabel('risk logit (gradient-decoupled hooks)')
ax.set_title(f'Forward bit-exactness   (max |Δ| = {max_err:.1e})')
plt.tight_layout(); plt.show()

## 7. SHAP-style per-token drivers, one panel per outcome

Per-token contributions to each outcome's risk logit, **computed on the deployed model** via the gradient-decoupled probe defined in §6. For each (patient, outcome) we run one forward pass with the decoupled hooks attached (forward = real) and back-propagate the risk logit to each block's transcoder features. The signed contribution of token $t$ to outcome $k$ is

$$\text{contrib}_t^{(k)} \;=\; \sum_{L=1}^{4} \sum_{F} f_L(t, F) \cdot \partial \mathrm{risk}_k / \partial f_L(t, F)\,.$$

Red bars push risk up, blue down. `_START` / `_END` interval markers and the `_BITZUA` site marker are stripped from labels; `[NULL]` and `ADMISSION_EVENT` are filtered out (no semantic content). Cost is one forward + K backwards per patient (down from the previous version's $L$ forwards), since one decoupled forward feeds attribution to all layers at once.

In [ ]:
import re
from collections import defaultdict
from xai.attribute import TranscoderHookManager

# Clear any hooks left from §6 (or a partially-failed earlier §7 run) so the
# attribution pass below sees a clean encoder.  Each per-patient call inside
# `attribute_all_outcomes_decoupled` opens + closes its own manager via the
# try/finally; this just guards against stale state across cell re-runs.
clear_mlp_hooks(model)

# ----------------------------- knobs --------------------------------------- #
# Gradient-decoupled attribution is one forward + K backwards per patient
# (cheaper than the per-layer-swap version), so we keep the scan at 300.
N_PATIENTS_SCAN  = 300
TOP_K_PER_SIDE   = 5            # cap per side (we show fewer if fewer real drivers exist)
MIN_FIRINGS      = 5            # drop tokens seen < this many times across the scan
EXCLUDE_TOKENS   = {'[NULL]', 'ADMISSION_EVENT'}  # excluded by exact concept name (no semantic content).
# Data-redundancy guard: source rows where the measured value was missing
# get a NaN suffix burned into the concept name at tokenization time
# (e.g. `ALANINE_AMINOTRANSFERASE_MEASURE_TREND_nan`).  These add no signal
# beyond "value was absent" and would otherwise dominate panels for rare
# outcomes.  Match the suffix case-insensitively, after `_START`/`_END` /
# `_BITZUA` stripping has already happened.
_NAN_SUFFIX_RE = re.compile(r'_(nan|none|null)$', re.IGNORECASE)

# ---------------------- fix id2concept (BUGFIX) ---------------------------- #
# tokenizer.id2token is already a Dict[int, str]; previous code used
# `enumerate(dict)` which iterates keys, so we mapped 0->0, 1->1 ... and
# every label fell back to the `id=N` fallback.
id2concept = dict(model.embedder.tokenizer.id2token)

_SE_RE     = re.compile(r'_(START|END)$')
# Strip `_BITZUA` from axis labels -- the marker would otherwise leak the
# source vocabulary and break double-blind for the paper submission.
_BITZUA_RE = re.compile(r'_BITZUA')
def canon_token(name):
    if name is None: return '[NULL]'
    s = _SE_RE.sub('', str(name))
    s = _BITZUA_RE.sub('', s)
    return s
id2canon = {cid: canon_token(name) for cid, name in id2concept.items()}

# Pre-compute the set of concept IDs whose canonical name carries a NaN /
# None / Null suffix.  Single set membership check in the inner loop.
EXCLUDE_NAN_CONCEPT_IDS = {
    cid for cid, canon in id2canon.items()
    if _NAN_SUFFIX_RE.search(canon)
}
print(f'NaN-suffix exclusion: skipping {len(EXCLUDE_NAN_CONCEPT_IDS)} concept(s).')
# Surface a handful of examples so you can sanity-check the regex caught the
# right thing (the suffix can appear in `_TREND_nan`, `_STATE_nan`, etc.).
_examples = [
    id2canon[cid] for cid in list(EXCLUDE_NAN_CONCEPT_IDS)[:8]
]
if _examples:
    print(f'  e.g. {_examples}')

risk_idx_cpu  = model.task_heads.risk_idx.cpu().tolist()
risk_outcomes = [model.outcome_names[i] for i in risk_idx_cpu]
print(f'Will explain {len(risk_outcomes)} outcomes: {risk_outcomes}')

# per_outcome_agg[oc_name] -> { canonical_token: [sum_signed_attrib, n_fires] }
per_outcome_agg = {oc: defaultdict(lambda: [0.0, 0]) for oc in risk_outcomes}


def attribute_all_outcomes_decoupled(batch, patient_idx):
    '''
    Gradient-decoupled attribution against the deployed model.

    One forward pass with gradient-decoupled hooks attached (forward = real,
    bit-exact); for each outcome k, backprop the risk logit to every block's
    transcoder features.  Per-token contribution for outcome k is

        contrib_t^k = sum_L sum_F f_L(p, t, F) * grad_L(p, t, F)

    Nothing in this expression depends on swap fidelity -- attribution lives
    on the real model.
    '''
    prev_ckpt = getattr(model, 'use_checkpoint', False)
    model.use_checkpoint = False
    K = len(risk_outcomes)
    mgr = TranscoderHookManager(
        model, transcoders, gradient_decoupled=True, enabled=True,
    ).attach(track_grad=True)
    try:
        risk_logits, _, _, pad_mask = model.predict(
            parent_raw_ids=batch['parent_raw_ids'],
            concept_ids=batch['concept_ids'],
            value_ids=batch['value_ids'],
            position_ids=batch['position_ids'],
            abs_ts=batch['abs_ts'],
            context_vec=batch['context_vec'],
        )
        leaves   = list(mgr.features.values())   # one per layer, all leaves
        pad_p    = pad_mask[patient_idx].detach().cpu()
        out      = []
        for k in range(K):
            target = risk_logits[patient_idx, k]
            grads = torch.autograd.grad(
                target, leaves,
                retain_graph=(k < K - 1),
                allow_unused=False,
            )
            # Sum contributions over (layer, feature) -> per-token signed contribution.
            per_tok = None
            for f, g in zip(leaves, grads):
                contrib = (f[patient_idx] * g[patient_idx]).detach().sum(dim=-1)  # [T]
                per_tok = contrib if per_tok is None else per_tok + contrib
            out.append((risk_outcomes[k], per_tok.cpu(), pad_p))
        return out
    finally:
        mgr.detach()
        model.use_checkpoint = prev_ckpt


# ------------------------ scan + aggregate --------------------------------- #
print(f'Scanning {N_PATIENTS_SCAN} patients (gradient-decoupled attribution)...')
pid_seen = 0
batch_iter = iter(dl)
pbar = tqdm(total=N_PATIENTS_SCAN, desc='patients', leave=False)
while pid_seen < N_PATIENTS_SCAN:
    try:
        batch = next(batch_iter)
    except StopIteration:
        batch_iter = iter(dl); batch = next(batch_iter)
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    B = batch['concept_ids'].size(0)
    for p in range(B):
        if pid_seen >= N_PATIENTS_SCAN: break
        results = attribute_all_outcomes_decoupled(batch, p)
        cids = batch['concept_ids'][p].cpu().tolist()
        for oc_name, per_tok, pad_p in results:
            agg = per_outcome_agg[oc_name]
            for t, keep in enumerate(pad_p.tolist()):
                if not keep: continue
                # Drop tokens whose CONCEPT name carries a NaN/None/Null
                # suffix -- these are dataset redundancy (value-missing rows
                # got that string burned into the concept at tokenization).
                if cids[t] in EXCLUDE_NAN_CONCEPT_IDS: continue
                name = id2canon.get(cids[t], f'id={cids[t]}')
                agg[name][0] += float(per_tok[t].item())
                agg[name][1] += 1
        pid_seen += 1; pbar.update(1)
pbar.close()

# --------------------------- grid plot ------------------------------------- #
n_oc  = len(risk_outcomes)
ncols = 2
nrows = (n_oc + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 1.6 * (2 * TOP_K_PER_SIDE) * 0.5 + 1))
axes = np.atleast_2d(axes).ravel()

for ax, oc_name in zip(axes, risk_outcomes):
    agg = per_outcome_agg[oc_name]
    rows_all = [
        (name, total / n, n)
        for name, (total, n) in agg.items()
        if n >= MIN_FIRINGS and name not in EXCLUDE_TOKENS
    ]
    # Up-to-K bars per side -- only count real positive / negative drivers.
    pos = [r for r in sorted(rows_all, key=lambda r: -r[1]) if r[1] > 0][:TOP_K_PER_SIDE]
    neg = [r for r in sorted(rows_all, key=lambda r:  r[1]) if r[1] < 0][:TOP_K_PER_SIDE]
    combined = pos[::-1] + neg
    if not combined:
        ax.set_title(oc_name + '  (no signal)', fontsize=9.5, loc='left')
        ax.set_visible(True); ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values(): s.set_visible(False)
        continue
    labels = [r[0] for r in combined]
    means  = np.array([r[1] for r in combined])
    counts = [r[2] for r in combined]
    colors = ['#e63946' if m > 0 else '#3a86ff' for m in means]
    y = np.arange(len(labels))
    ax.barh(y, means, color=colors, edgecolor='white', linewidth=0.6)
    ax.axvline(0, color='#333', lw=0.6)
    ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=7.5)
    ax.invert_yaxis()
    ax.spines['left'].set_visible(False); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.grid(axis='x', alpha=0.25)
    ax.set_title(oc_name, fontsize=10, loc='left', pad=4)
    ax.axhline(len(pos) - 0.5, color='#bbb', lw=0.6, linestyle='--')

for ax in axes[n_oc:]:
    ax.set_visible(False)

fig.suptitle(f'Per-token risk drivers, one panel per outcome  ·  averaged over {pid_seen} patients\n'
             'gradient-decoupled attribution through the deployed model   ·   '
             'red = pushes risk up    ·    blue = pushes risk down',
             fontsize=11)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()